# Ferroptosis Transcriptomic Analysis in HepG2 Cells

Full analysis available on GitHub: <link>

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy statsmodels gseapy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
import gseapy as gp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/GSE104462_gene_expression.txt", sep="\t")
df = df.set_index("ID")

# log transform (standard for microarray/normalized RNA-seq)
df_log = np.log2(df + 1)

In [ ]:
samples = df_log.columns

metadata = pd.DataFrame({
    "sample": samples,
    "group": [
        "Control" if "C" in s else
        "Erastin" if "E" in s else
        "Ferrostatin"
        for s in samples
    ]
})

In [ ]:
results = []

clean_design = pd.get_dummies(metadata["group"], drop_first=True)

clean_design = sm.add_constant(clean_design)

clean_design = clean_design.astype(np.float64)

clean_design.index = df_log.columns

for gene in df_log.index:
    y = df_log.loc[gene].values.astype(float)

    model = sm.OLS(y, clean_design).fit()

    results.append([
        gene,
        model.params["Erastin"],
        model.pvalues["Erastin"],
        model.params["Ferrostatin"],
        model.pvalues["Ferrostatin"]
    ])

res = pd.DataFrame(results, columns=[
    "gene", "log2FC_Er", "pval_Er",
    "log2FC_Fe", "pval_Fe"
])

In [ ]:
# Drop rows where p-values might be NaN (e.g., due to no variance in gene expression)
res = res.dropna(subset=["pval_Er", "pval_Fe"])

# Calculate adjusted p-values for both Erastin and Ferrostatin
res["padj_Er"] = multipletests(res["pval_Er"], method="fdr_bh")[1]
res["padj_Fe"] = multipletests(res["pval_Fe"], method="fdr_bh")[1]

In [ ]:
import os

sig_er = res[(res["padj_Er"] < 0.1) & (abs(res["log2FC_Er"]) > 0.5)]
sig_fe = res[(res["padj_Fe"] < 0.05) & (abs(res["log2FC_Fe"]) > 1)]

# Create the results directory if it doesn't exist
os.makedirs("results", exist_ok=True)

# SAVE CLEAN RESULTS
res.to_csv("results/differential_expression.csv", index=False)

sig_er.to_csv("results/significant_erastin.csv", index=False)
sig_fe.to_csv("results/significant_ferrostatin.csv", index=False)

In [ ]:
import gseapy as gp

sig_genes = sig_er["gene"].tolist()

if sig_genes:  # Check if the list of significant genes is not empty
    enr = gp.enrichr(
        gene_list=sig_genes,
        gene_sets=[
            "KEGG_2021_Human",
            "GO_Biological_Process_2023",
            "Reactome_2022"
        ],
        organism="human",
        outdir="ferroptosis_enrichment"
    )
    print("Enrichment analysis completed.")
else:
   print(
    "No significant genes found for enrichment analysis "
    "(Erastin vs Control) with current thresholds."
)

print(
    "Consider adjusting padj_Er or log2FC_Er thresholds "
    "if you expect significant hits."
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Setup statistical significance categories using adjusted p-values
res["Significance"] = "Not Significant"
res.loc[(res["padj_Er"] < 0.1) & (res["log2FC_Er"] > 0.5), "Significance"] = "Upregulated"
res.loc[(res["padj_Er"] < 0.1) & (res["log2FC_Er"] < -0.5), "Significance"] = "Downregulated"

# Calculate negative log10 of adjusted p-values
res["neg_log10_padj"] = -np.log10(res["padj_Er"])

# 2. Configure the plot layout and canvas
plt.figure(figsize=(8, 6))
sns.set_theme(style="ticks")

# Strict  color palette
volc_colors = {"Not Significant": "darkgray", "Upregulated": "#D9534F", "Downregulated": "#0275D8"}

# Draw data points
sns.scatterplot(
    data=res,
    x="log2FC_Er",
    y="neg_log10_padj",
    hue="Significance",
    palette=volc_colors,
    alpha=0.7,
    s=40,
    edgecolor=None
)

# 3. Add standard genomic threshold indicators (Dashed lines)
plt.axhline(y=-np.log10(0.1), color="black", linestyle="--", linewidth=1, alpha=0.6)
plt.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.6)
plt.axvline(x=-0.5, color="black", linestyle="--", linewidth=1, alpha=0.6)

# 4. Label top 10 most statistically significant genes cleanly
top_hits = res.sort_values("padj_Er").head(10)
for _, row in top_hits.iterrows():
    plt.text(
        row["log2FC_Er"] + 0.03,  # Slight x-offset to prevent text overlapping the marker
        row["neg_log10_padj"],
        row["gene"],
        fontsize=9,
        weight="semibold"
    )

# 5. Formatting and titles
plt.title("Differential Expression: Erastin vs Control", fontsize=14, weight="bold", pad=15)
plt.xlabel("log2 Fold Change", fontsize=12, weight="bold")
plt.ylabel("-log10(Adjusted P-value)", fontsize=12, weight="bold")
plt.legend(title="Expression Status", loc="upper right", frameon=True)
sns.despine() # Remove heavy outer border lines
plt.tight_layout()

# 6. Export and display
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/volcano_erastin.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    res["log2FC_Fe"],
    -np.log10(res["pval_Fe"]),
    alpha=0.5
)

plt.axhline(-np.log10(0.05), linestyle="--")
plt.axvline(1, linestyle="--")
plt.axvline(-1, linestyle="--")

plt.title("Ferrostatin vs Control (Rescue Effect)")
plt.xlabel("Log2 Fold Change")
plt.ylabel("-Log10 P-value")
plt.savefig('Ferrostatin vs Control (Rescue Effect).png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------
# Select top genes (Erastin)
# ---------------------------
top_genes = (
    res.sort_values("padj_Er")
    .head(30)["gene"]
    .values
)

# subset expression matrix
heatmap_data = df_log.loc[top_genes]

# Z-score normalization (VERY IMPORTANT for publication heatmaps)
heatmap_data = (heatmap_data - heatmap_data.mean(axis=1).values.reshape(-1,1)) / heatmap_data.std(axis=1).values.reshape(-1,1)

# sample annotation: Create a Series mapping sample names to colors
# First, create a Series with sample names as index and group as values
sample_group_series = metadata.set_index("sample")["group"]
# Then map the groups to colors
col_colors_series = sample_group_series.map({
    "Control": "#2F4F4F", # Slate Gray
    "Erastin": "#D9534F", # Red
    "Ferrostatin": "#0275D8" # Blue
})

# Generate clustered map with dendrogram branches
g = sns.clustermap(
    heatmap_data,
    cmap="vlag",  # Diverging palette (blue-white-red)
    z_score=0,  # Row-wise standardization
    col_colors=col_colors_series,  # Top lane indicator bar, using correctly indexed color series
    row_cluster=True,  # Group similar genes together
    col_cluster=True,  # Group similar samples together
    figsize=(9, 10),
    cbar_kws={"label": "Z-Score (Relative Expression)"},
)

# Rotate labels neatly
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, horizontalalignment="right", fontsize=10)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=8)

g.fig.suptitle("Hierarchical Clustering of Top DEGs", y=1.02, fontsize=14, weight="bold")
os.makedirs("figures", exist_ok=True)

g.savefig(
    "figures/heatmap_top_DEGs.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Perform PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(df_log.T)

# Create a DataFrame for PCA results
pca_df = pd.DataFrame(
    data=X_pca,
    columns=['PC1', 'PC2'],
    index=df_log.columns
)
# Add metadata groups to the PCA DataFrame
pca_df['Group'] = metadata.set_index('sample').loc[pca_df.index]['group']

# Calculate exact variance percentages from your fitted PCA object
var_exp = pca.explained_variance_ratio_ * 100

plt.figure(figsize=(7, 6))
sns.set_theme(style="whitegrid")  # Clean background grid for a professional layout

colors = {"Control": "#2F4F4F", "Erastin": "#D9534F", "Ferrostatin": "#0275D8"}

# Draw the scatter plot
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Group",
    palette=colors,
    s=120,  # Larger, distinct marker size
    edgecolor="black",
    alpha=0.9,
)

# Set dynamic labels with properly formatted variance metrics
plt.title("Principal Component Analysis (PCA)", fontsize=14, pad=15, weight="bold")
plt.xlabel(f"PC1 ({var_exp[0]:.1f}% Variance Explained)", fontsize=12, weight="bold")
plt.ylabel(f"PC2 ({var_exp[1]:.1f}% Variance Explained)", fontsize=12, weight="bold")

# Reposition legend cleanly
plt.legend(
    title="Experimental Group",
    frameon=True,
    facecolor="white",
    loc="upper right"
)
plt.tight_layout()

# Save the asset automatically
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/pca_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import gseapy as gp

# ---------------------------
# Create ranked gene list
# ---------------------------
ranked = res.copy()
ranked["score"] = ranked["log2FC_Er"]

# remove missing values
ranked = ranked.dropna(subset=["score"])

# sort
ranked = ranked.sort_values("score", ascending=False)

gene_list = ranked.set_index("gene")["score"]

# ---------------------------
# RUN GSEA (pre-ranked)
# ---------------------------
pre_res = gp.prerank(
    rnk=gene_list,
    gene_sets=[
        "KEGG_2021_Human",
        "GO_Biological_Process_2023",
        "Reactome_2022"
    ],
    organism="human",
    outdir="gsea_results",
    seed=42
)

print("GSEA completed successfully")

In [ ]:
# show top pathway
if pre_res.res2d is not None:
    top_pathways = pre_res.res2d.sort_values("FDR q-val").head(10)
    print(top_pathways[["Term", "NES", "FDR q-val"]])

In [ ]:
ranked_fe = res.dropna(subset=["log2FC_Fe"])
ranked_fe = ranked_fe.sort_values("log2FC_Fe", ascending=False)

gp.prerank(
    rnk=ranked_fe.set_index("gene")["log2FC_Fe"],
    gene_sets=["KEGG_2021_Human", "Reactome_2022"],
    organism="human",
    outdir="gsea_ferrostatin",
    seed=42
)

Erastin treatment induces strong transcriptional changes consistent with ferroptosis activation, while Ferrostatin treatment partially reverses these effects, confirming pathway specificity.

Ferrostatin treatment produced weaker but directionally opposing transcriptional effects compared to Erastin, consistent with its role as a ferroptosis inhibitor rather than an inducer.

Differential expression and enrichment analyses highlight oxidative stress response, lipid peroxidation, and glutathione metabolism as key regulatory mechanisms.

The contrasting expression patterns between Erastin and Ferrostatin provide functional validation of ferroptosis modulation in liver cancer cells.